Fixed structures - AutoCAD insert script
=
This notebook writes an AutoCAD script that inserts one fixed-structure block per table, from a list of string positions. Single-string tables get `EMF302_estrutura_fixa_1`, two-high tables get `EMF302_estrutura_fixa_2`.

**Workflow overview:**
1. Read the string list (`Type`, `Position X`, `Position Y`) from Excel.
2. Build one `-insert` line per `C` (single) and per `T` (top) string. `B` (bottom) strings get none, since the `T` string of the same table already places the two-high structure.
3. Write the lines to `Script Estruturas.scr` on the Desktop.

**Input prerequisites:**
- **File:** `C:\Users\Usuario\Downloads\Strings.xlsx`, first sheet.
- **Columns:** `Type`, `Position X`, `Position Y` (exact names, case-sensitive).
- **`Type` values:** `C` (single string on the table), `T` (top string of a two-high table), `B` (bottom string). Any other value gets no structure.
- **Positions:** the string centre. For `T`, the table centre is 1.1513 m below the top string.
- **No empty cells** in any column of a `C` / `T` row: `dropna()` drops the whole row, not only the `B` rows.

**Running the `.scr` in AutoCAD:**
- The blocks `EMF302_estrutura_fixa_1` and `EMF302_estrutura_fixa_2` must already be defined in the target drawing.
- Each line passes `1 0` after the point (one scale factor, then rotation 0). The blocks must have *Scale uniformly* on (only one scale prompt) and no attributes to prompt for, otherwise the values go into the wrong prompts.

Imports
=
pandas for the Excel file, numpy for `np.where`.

In [10]:
import pandas as pd

import numpy as np

Building and writing the insert script
=
- `Script` starts as `None` for every row.
- Nested `np.where` on `Type`:
  - `C` → `-insert EMF302_estrutura_fixa_1 <X>,<Y> 1 0`
  - `T` → `-insert EMF302_estrutura_fixa_2 <X>,<Y - 1.1513> 1 0` (`dif` = offset from the top string to the table centre).
  - anything else (`B`) → keeps `None`.
- `dropna()` removes the rows with no script. **Warning:** it looks at every column, so a `C`/`T` row with any empty cell is dropped too and its structure is missing from the script.
- Writes one command per line to `C:\Users\Usuario\Desktop\Script Estruturas.scr` (hardcoded path, overwritten). The spaces in each line act as Enter in the script.

In [13]:

df_strings = pd.read_excel(r"C:\Users\Usuario\Downloads\Strings.xlsx")
df_strings["Script"] = None

dif = 1.1513

df_strings["Script"] = np.where(
    (df_strings["Type"] == "C"),
    "-insert EMF302_estrutura_fixa_1 " + \
    df_strings['Position X'].astype(str) + "," + \
    df_strings['Position Y'].astype(str) + \
    " 1 0",
    np.where(
        (df_strings["Type"] == "T"),
        "-insert EMF302_estrutura_fixa_2 " + \
        df_strings['Position X'].astype(str) + "," + \
        (df_strings['Position Y']-dif).astype(str) + \
        " 1 0",
        df_strings["Script"]
    )
)

df_strings = df_strings.dropna()

with open(r"C:\Users\Usuario\Desktop\Script Estruturas.scr", 'w') as f:
    f.write('\n'.join(df_strings["Script"].astype(str)) + '\n')


Empty cell
=
Nothing to run here.